In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Kunci latihan

Jalankan dari atas ke bawah. Angka yang tercetak disalin ke README.


In [2]:
# Pustaka adalah kumpulan fungsi siap pakai, bukan rak buku.
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.tree import DecisionTreeClassifier


In [3]:
# Satu ide: rata-rata seluruh nilai latihan.
nilai = np.array([1, 3, 5, 7])
print("latihan_mean_semua", float(nilai.mean()))
assert float(nilai.mean()) == (1 + 3 + 5 + 7) / 4


latihan_mean_semua 4.0


In [4]:
# Satu ide: rata-rata dua nilai data latih.
print("latihan_mean_latih", float(nilai[:2].mean()))
assert float(nilai[:2].mean()) == (1 + 3) / 2


latihan_mean_latih 2.0


In [5]:
# Satu ide: rata-rata yang boleh dipakai.
print("latihan_rata_boleh", "data latih")


latihan_rata_boleh data latih


In [6]:
# Satu ide: jam yang dikocok memasukkan masa depan ke data latih.
jam = np.arange(12)
acak_latih, acak_uji = train_test_split(
    jam, test_size=0.25, random_state=SEED, shuffle=True
)
print("latihan_acak_uji_min", int(acak_uji.min()))
print("latihan_acak_latih_max", int(acak_latih.max()))
print(
    "latihan_masa_depan_di_latih",
    int(np.sum(acak_latih > acak_uji.min())),
)
assert int(np.sum(acak_latih > acak_uji.min())) > 0


latihan_acak_uji_min 0
latihan_acak_latih_max 11
latihan_masa_depan_di_latih 9


In [7]:
# Satu ide: giliran yang menahan jam belakangan tetap urut.
# TimeSeriesSplit memotong data berurutan tanpa mengocok. Bukan istilah baru.
pembagi = TimeSeriesSplit(n_splits=2)
for i, (bagian_latih, bagian_uji) in enumerate(pembagi.split(jam)):
    print(
        "latihan_giliran",
        i,
        "latih_akhir",
        int(bagian_latih.max()),
        "uji_awal",
        int(bagian_uji.min()),
        "urut",
        int(bagian_latih.max() < bagian_uji.min()),
    )
    assert bagian_latih.max() < bagian_uji.min()


latihan_giliran 0 latih_akhir 3 uji_awal 4 urut 1
latihan_giliran 1 latih_akhir 7 uji_awal 8 urut 1


In [8]:
# Satu ide: catatan jam tidak dikocok.
print("latihan_boleh_kocok", "tidak")


latihan_boleh_kocok tidak


In [9]:
# Satu ide: bagi data latihan pohon. test_size=0.3 menyisihkan 30 persen baris.
# Menurut dokumentasi scikit-learn, make_classification mengundi soal klasifikasi.
# n_samples adalah banyaknya baris.
# n_features adalah banyaknya kolom semuanya: kolom informatif, kolom redundan, kolom salinan, dan sisanya kolom acak yang tidak berguna.
# n_informative adalah banyaknya kolom informatif. Tiap kelas punya kelompok sebaran normal (simpangan baku 1) di sekitar sudut hypercube berdimensi sebanyak itu.
# Hypercube adalah kotak bersudut di beberapa dimensi. Bukan istilah baru.
# Di dalam tiap kelompok, kolom informatif diundi dari sebaran normal baku (rata-rata 0, simpangan baku 1), lalu digabung secara linear, sebelum kelompok diletakkan di sudut hypercube.
# n_redundant adalah banyaknya kolom kombinasi linear acak dari kolom informatif.
# n_repeated adalah banyaknya kolom salinan, diambil acak dari kolom informatif dan redundan.
# n_clusters_per_class adalah banyaknya kelompok pada tiap kelas.
# class_sep mengalikan ukuran hypercube. Panjang sisinya 2 kali class_sep. Makin besar, kelompok makin berjauhan dan soal makin mudah.
# flip_y adalah bagian baris yang kelasnya diundi ulang secara acak.
# random_state membuat undian pembuatan data ini berulang selama angkanya sama.
# Nama parameter ini bukan istilah baru.
X, y = make_classification(
    n_samples=300,
    n_features=25,
    n_informative=2,
    n_redundant=0,
    n_repeated=0,
    n_clusters_per_class=1,
    class_sep=0.6,
    flip_y=0.12,
    random_state=SEED,
)
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.3, random_state=SEED, shuffle=True
)
print("latihan_n_latih", int(len(y_latih)))
print("latihan_n_uji", int(len(y_uji)))


latihan_n_latih 210
latihan_n_uji 90


In [10]:
# Satu ide: skor pohon tanpa batas pada data latih.
pohon_dalam = DecisionTreeClassifier(max_depth=None, random_state=SEED)
pohon_dalam.fit(X_latih, y_latih)
print("latihan_skor_latih_dalam", float(pohon_dalam.score(X_latih, y_latih)))


latihan_skor_latih_dalam 1.0


In [11]:
# Satu ide: skor pohon tanpa batas pada data uji.
print("latihan_skor_uji_dalam", float(pohon_dalam.score(X_uji, y_uji)))


latihan_skor_uji_dalam 0.7333333333333333


In [12]:
# Satu ide: skor pohon pendek pada data latih.
pohon_dangkal = DecisionTreeClassifier(max_depth=2, random_state=SEED)
pohon_dangkal.fit(X_latih, y_latih)
print(
    "latihan_skor_latih_dangkal",
    float(pohon_dangkal.score(X_latih, y_latih)),
)


latihan_skor_latih_dangkal 0.8380952380952381


In [13]:
# Satu ide: skor pohon pendek pada data uji.
print("latihan_skor_uji_dangkal", float(pohon_dangkal.score(X_uji, y_uji)))


latihan_skor_uji_dangkal 0.8111111111111111


In [14]:
# Satu ide: pohon dengan selisih skor lebih besar adalah yang menghafal.
selisih_dalam = pohon_dalam.score(X_latih, y_latih) - pohon_dalam.score(
    X_uji, y_uji
)
selisih_dangkal = pohon_dangkal.score(X_latih, y_latih) - pohon_dangkal.score(
    X_uji, y_uji
)
print("latihan_selisih_dalam", float(selisih_dalam))
print("latihan_selisih_dangkal", float(selisih_dangkal))
if selisih_dalam > selisih_dangkal:
    print("latihan_menghafal", "dalam")
else:
    print("latihan_menghafal", "dangkal")
assert selisih_dalam > selisih_dangkal


latihan_selisih_dalam 0.2666666666666667
latihan_selisih_dangkal 0.026984126984127
latihan_menghafal dalam
